<a href="https://colab.research.google.com/github/niola33/OutamationWork-/blob/main/Updated_researchchatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gradio pymupdf pytesseract pillow sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.4 MB/s eta 0:00:00


In [2]:
import fitz
import gradio as gr
import numpy as np
import faiss
import pytesseract
import torch
import time
import json

from PIL import Image
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [3]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

if torch.cuda.is_available():
    model = model.to("cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [4]:
chunks = []
metadata = []
index = None

chat_history = []
feedback_log = []

In [5]:
def extract_text(pdf_path):

    doc = fitz.open(pdf_path)
    pages = []

    for i in range(len(doc)):
        page = doc[i]
        text = page.get_text()

        if not text.strip():

            pix = page.get_pixmap()
            img = Image.frombytes("RGB",[pix.width,pix.height],pix.samples)
            text = pytesseract.image_to_string(img)

        pages.append((i+1,text))

    return pages

In [6]:
def chunk_text(text, size=500, overlap=100):

    pieces=[]
    start=0

    while start < len(text):
        end=start+size
        pieces.append(text[start:end])
        start += size-overlap

    return pieces

In [7]:
def process_pdf(file):

    global chunks,metadata,index

    start=time.time()

    chunks=[]
    metadata=[]

    pages = extract_text(file.name)

    for page_num,text in pages:

        pieces = chunk_text(text)

        for c in pieces:

            chunks.append(c)

            metadata.append({
                "page":page_num,
                "source":file.name
            })

    embeddings = embed_model.encode(chunks)

    dim = embeddings.shape[1]

    index = faiss.IndexFlatL2(dim)
    index.add(np.array(embeddings))

    end=time.time()

    return f"Document processed: {len(pages)} pages, {len(chunks)} chunks in {round(end-start,2)} seconds."

In [8]:
def retrieve(query,k=4):

    query_vec = embed_model.encode([query])

    distances,ids = index.search(np.array(query_vec),k)

    results=[]

    for i in ids[0]:

        results.append({
            "text":chunks[i],
            "meta":metadata[i]
        })

    return results

In [9]:
def build_prompt(query,retrieved):

    context=""

    for r in retrieved:
        context+=f"[Page {r['meta']['page']}]\n{r['text']}\n\n"

    prompt=f"""
You are a research assistant helping analyze an academic research paper.

Use ONLY the provided context.

Tasks:
1. Answer the question clearly.
2. Summarize the key findings.
3. Provide main bullet points if appropriate.
4. Cite page numbers.

Context:
{context}

Question:
{query}

Answer:
"""

    return prompt

In [10]:
def generate_answer(prompt):

    inputs = tokenizer(prompt,return_tensors="pt",truncation=True,max_length=2048)

    if torch.cuda.is_available():
        inputs = {k:v.to("cuda") for k,v in inputs.items()}

    outputs = model.generate(**inputs,max_new_tokens=300)

    return tokenizer.decode(outputs[0],skip_special_tokens=True)

In [11]:
def chat_fn(message,history):

    start=time.time()

    retrieved = retrieve(message)

    prompt = build_prompt(message,retrieved)

    answer = generate_answer(prompt)

    sources=[f"Page {r['meta']['page']}" for r in retrieved]

    end=time.time()

    response_time=round(end-start,2)

    formatted=f"""
{answer}

Sources: {sources}

Response time: {response_time} seconds
"""

    history = history + [(message,formatted)]

    chat_history.append({
        "question":message,
        "answer":answer,
        "sources":sources,
        "time":response_time
    })

    return history,""

In [12]:
def record_feedback(label):

    if not chat_history:
        return "No answer yet."

    feedback_log.append({
        "answer":chat_history[-1],
        "rating":label
    })

    return f"Feedback recorded: {label}"

In [13]:
def download_answers():

    if not chat_history:
        return None

    file="answers.json"

    with open(file,"w") as f:
        json.dump(chat_history,f,indent=2)

    return file

In [14]:
with gr.Blocks(
    title="Research Paper Q&A Assistant",
    theme=gr.themes.Soft(primary_hue="green"),
    css="""
    body{
        background:linear-gradient(135deg,#0b1f14,#123524,#1b4332);
    }

    .panel{
        background:rgba(255,255,255,0.06);
        border-radius:18px;
        box-shadow:0 10px 25px rgba(0,0,0,0.4);
        padding:18px;
    }

    button{
        border-radius:12px;
        box-shadow:0 6px 18px rgba(0,0,0,0.3);
    }
    """
) as demo:

    gr.Markdown("# 📚 Research Paper Q&A Assistant")

    with gr.Row():

        with gr.Column(scale=1,elem_classes="panel"):

            pdf = gr.File(label="Upload Research Paper PDF")

            process_btn = gr.Button("Process Paper")

            status = gr.Textbox(label="Processing Status")

            download_btn = gr.Button("Download Answers")

            download_file = gr.File()

        with gr.Column(scale=2,elem_classes="panel"):

            chatbot = gr.Chatbot(height=520)

            msg = gr.Textbox(
                placeholder="Ask questions about the research paper..."
            )

            send = gr.Button("Ask Question")

            gr.Markdown("### Was this answer helpful?")

            with gr.Row():
                thumbs_up = gr.Button("👍")
                thumbs_down = gr.Button("👎")

            feedback_status = gr.Textbox(label="Feedback Status")

    process_btn.click(
        process_pdf,
        inputs=pdf,
        outputs=status
    )

    send.click(
        chat_fn,
        inputs=[msg,chatbot],
        outputs=[chatbot,msg]
    )

    msg.submit(
        chat_fn,
        inputs=[msg,chatbot],
        outputs=[chatbot,msg]
    )

    thumbs_up.click(
        lambda:record_feedback("good"),
        outputs=feedback_status
    )

    thumbs_down.click(
        lambda:record_feedback("bad"),
        outputs=feedback_status
    )

    download_btn.click(
        download_answers,
        outputs=download_file
    )

demo.launch(share=True)

/tmp/ipykernel_695/660371715.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_695/660371715.py:1: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_695/660371715.py:41: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=520)
/tmp/ipykernel_695/660371715.py:41: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if yo

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cabcbdf87190b7460f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
